In [3]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "unlearn" / "data.py").is_file():
    repo_root = repo_root.parent

src_dir = repo_root / "src"
if not (src_dir / "unlearn" / "data.py").is_file():
    raise FileNotFoundError(f"Could not find src/unlearn/data.py from {Path.cwd()}")
sys.path.insert(0, str(src_dir))

from unlearn.device import DEVICE
from unlearn.data import get_cifar10_loaders
from unlearn.models import load_cifar10_model
from unlearn.metrics import accuracy, mia_accuracy
from unlearn.methods import finetune

print("Running on device:", str(DEVICE).upper())

Running on device: MPS


In [4]:
# Load CIFAR-10 and create the train, test, validation, forget, and retain loaders.
loaders = get_cifar10_loaders()
train_loader, test_loader, val_loader = loaders["train"], loaders["test"], loaders["val"]
forget_loader, retain_loader = loaders["forget"], loaders["retain"]

In [5]:
model = load_cifar10_model(repo_root / "weights_resnet18_cifar10.pth")

In [6]:
# Load the challenge's CIFAR-10-pretrained ResNet-18.
import copy

ft_model = finetune.unlearning(copy.deepcopy(model), retain_loader, forget_loader, val_loader)

print(f"Retain acc: {100*accuracy(ft_model, retain_loader):.1f}%")
print(f"Test acc:   {100*accuracy(ft_model, test_loader):.1f}%")
print(f"MIA:        {mia_accuracy(ft_model, forget_loader, test_loader):.3f}")

Retain acc: 98.2%
Test acc:   83.6%
MIA:        0.509
